# MDF: the optimiser around a converged analysis

MDF (multidisciplinary feasible) is PROCESS's own architecture. VMCON owns the iteration
variables. Every time it evaluates the objective and the constraints, the models
underneath are converged first, by PROCESS's idempotence loop. So every point the
optimiser sees is a consistent machine.

The port keeps that architecture and changes the mechanics. PROCESS converges the whole
model sequence, and takes VMCON's gradient by finite differences with one full
evaluation per iteration variable. Here only the models that feed back into each other
are iterated, each group by its own driver. The gradient comes from automatic
differentiation through the converged analysis, in one call.

MDF takes three operations on the graph. Cut the feedback loops into fixed-point
problems. Insert the optimisation problem. Declare that every other problem is solved
inside the optimiser's iteration (`NestInside`). SAND answers the same structure the
other way: it folds every problem into the optimiser (`Combine`). IDF sits in between.
Each has its own notebook.

In [1]:
import os, sys, time
from pathlib import Path

HERE = Path.cwd()                                   # the notebook's own folder
REPO = next(p for p in (HERE, *HERE.parents) if (p / "functional_process").is_dir())
os.chdir(REPO)                                      # input files are named relative to PROCESS/
# The editable cottax checkout beside this repo, when the layout is the documented one
# (`two_opt_driver/scripts/README.md`), ahead of any installed copy.
_jaxgraph = REPO.parent.parent / "jaxgraph" / "src"
for entry in (str(REPO), str(_jaxgraph)):
    if Path(entry).is_dir() and entry not in sys.path:
        sys.path.insert(0, entry)

import jax
jax.config.update("jax_enable_x64", True)          # PROCESS is float64 throughout
import cottax
import numpy as np


print("repo  :", REPO)
print("cottax:", Path(cottax.__file__).parent)

repo  : /home/wrutten/projects/functional_PROCESS/PROCESS
cottax: /home/wrutten/projects/jaxgraph/src/cottax


## The graph

The models of the Helias stellarator input file, as the port declares them. Each node
is one model function; it reads and writes PROCESS's own variables (`.physics.rmajor`
is `data.physics.rmajor`). One disconnected node is left out. The file also poses the
problem: iteration variables `ixc`, constraints `icc`, figure of merit.
`native_reference` reads all of that off the file without running PROCESS.

In [2]:
INPUT = "tests/regression/input_files/stellarator_helias.IN.DAT"

import re

from cottax.blocking import Blocking
from cottax.plan import Plan
from cottax.problem import is_fixed_point

from functional_process.architecture_examples.notebook_tools import print_recipe
from functional_process.cottax import native
from functional_process.cottax.indat import graph_for, machine_from_indat, switch_values_from_indat
from functional_process.cottax.mda_harness import _without_excluded
from functional_process.cottax.queries import declared
from functional_process.cottax.visualization.grouping import driver_name, problem_kind

ref = native.native_reference(INPUT)
sw = switch_values_from_indat(INPUT)          # the static switch values the condition nodes are bound with
machine_graph = graph_for(machine_from_indat(INPUT))
raw = _without_excluded(machine_graph)

print(f"{len(raw.nodes)} nodes; cyclic components of sizes {[len(c) for c in raw.cycles]}")
print("problems the models declare themselves:")
for p in declared(raw):
    print(f"   {p.spelling:55s} {problem_kind(raw[p])}")
print(f"\nixc = {ref.ixc}")
print(f"icc = {ref.icc}  ({ref.n_equality} equalities), objective: figure of merit {ref.i_figure_merit}")

152 nodes; cyclic components of sizes [2, 6, 2, 2, 2]
problems the models declare themselves:
   ^problem.stellarator.coils.intersect                    root-find
   ^problem.physics.profiles.ion_vol_avg_temperature       fixed-point
   ^problem.power.delta_eta_step                           fixed-point

ixc = [2, 3, 4, 6, 10, 56, 59, 109]
icc = [2, 16, 24, 8, 17, 18, 67, 82, 83, 62, 32, 34, 35, 65]  (2 equalities), objective: figure of merit 6


## The recipe

### Step 1: cut the feedback loops

Each group of models that feed back into each other is opened with copies. `mda.CUTS`
names nine variables, picked by hand so that one iteration of the copies is one PROCESS
pass. `mda.cut_ops` turns the ones present in this graph into one operation per loop.
Each `FixedPointCut` gives the readers of a variable a copy, `^hat.x`, and adds the
requirement that the copy equals the computed value. That requirement is a fixed-point
problem. The `mda_gauss_seidel` notebook derives such cuts mechanically, and a recipe
from there can be used here instead.

In [3]:
from functional_process.cottax.mda import cut_graph, cut_ops

plan = Plan(raw)
for op in cut_ops(raw):
    plan = plan + op
print_recipe(plan)
print("\nsame as mda.cut_graph(raw):", plan.graph == cut_graph(raw))
minted = [p for p in declared(plan.graph) if p not in set(declared(raw))]
print("problems the cut minted:", [p.spelling for p in minted])

   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)

same as mda.cut_graph(raw): True
problems the cut minted: ['^problem.physics.proton_rate_density.cycle', '^problem.fwbs.f_ster_div_single']


### Step 2: state the file's problem

One node per active constraint and one for the figure of merit. They compute the same
values `constraints.py` and `objectives.py` do. Then the optimisation problem itself,
an `Optimise` node: minimise the objective over the iteration variables, subject to the
constraints. All are inserted into the graph like any other node. `sand.optimise_graph`
does the same; it is spelled out here so the step is visible.

In [4]:
from cottax.names import PathMap
from cottax.plan import Insert
from cottax.problem import Optimise
from cottax.spec import NodePath
from jax.tree_util import GetAttrKey

from functional_process.cottax.indat import objective_selection
from functional_process.cottax.sand import constraint_nodes, iteration_variable_path, objective_nodes, optimise_graph

nodes, equalities, inequalities, omitted = constraint_nodes(plan.graph, ref.icc, ref.n_equality, sw)
objective_built, objective = objective_nodes(plan.graph, objective_selection(ref.i_figure_merit), sw)
nodes.update(objective_built)
OPT = NodePath((GetAttrKey("Opt"),))
nodes[OPT] = Optimise(
    objective=objective,
    unknowns=tuple(iteration_variable_path(i) for i in ref.ixc),
    equalities=tuple(equalities),
    inequalities=tuple(inequalities),
)
plan = plan + Insert(PathMap(nodes.items()))
print_recipe(plan)
if omitted:
    print("constraints this port cannot state yet:", omitted)
reference_graph, _, _ = optimise_graph(cut_graph(raw), ref.ixc, ref.icc, ref.n_equality, ref.i_figure_merit, switch_values=sw)
print("same as sand.optimise_graph(...):", plan.graph == reference_graph)

   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)
   insert(.Constraint2, .Constraint16, .Constraint24, .Constraint8, .Constraint17, .Constraint18, .Constraint67,  ...
same as sand.optimise_graph(...): True


The optimisation problem reads the objective and the constraints. It also owns the
iteration variables, which almost everything reads. So it closes one big loop over most
of the machine. That loop now contains several problems. The blocking refuses it until
it is told how they relate, and the refusal names the two options: `Combine` and
`NestInside`. Choosing between them is choosing the architecture.

In [5]:
blocking = Blocking.scc(plan.graph)
big = max(blocking.blocks, key=len)
print(f"largest block: {len(big)} of {len(plan.graph.nodes)} nodes, holding",
      [p.spelling for p in declared(plan.graph) if p in set(big)], "\n")
try:
    blocking.problems
except ValueError as refusal:                 # the node list dropped from the message
    print(re.sub(r"block \(.*?\) declares", "the block declares", str(refusal), flags=re.S))

largest block: 123 of 170 nodes, holding ['^problem.stellarator.coils.intersect', '^problem.physics.profiles.ion_vol_avg_temperature', '^problem.physics.proton_rate_density.cycle', '^problem.fwbs.f_ster_div_single', '.Opt'] 



the block declares several problems ((NodePath(^problem.stellarator.coils.intersect), NodePath(^problem.physics.profiles.ion_vol_avg_temperature), NodePath(^problem.physics.proton_rate_density.cycle), NodePath(^problem.fwbs.f_ster_div_single), NodePath(.Opt))) with nothing saying which is outer -- one driver answers one problem, so `Combine` them into a single problem over every unknown, or `NestInside` one of them. Which is a modelling decision, not something the blocking can read off the graph


### Step 3: nest, so the optimiser is the outer loop

`NestInside(Opt)` records on the graph that every other problem in the loop is solved
inside the optimiser's iteration. Nothing is rewritten. The run order now has one
problem at the top level, the optimiser, and its interior is the whole analysis.
`mdf.nested_blocking` builds exactly this.

In [6]:
from cottax.rewrites import NestInside

from functional_process.cottax import mdf

plan = plan + NestInside(OPT)
print("the whole recipe:")
print_recipe(plan)

blocking = Blocking.scc(plan.graph)
i = blocking.index[OPT]
print("\nanswered at the top level:", [p.spelling for p in blocking.problems if p is not None])
inner = blocking.inner[i]
print(f"interior of the optimiser's block: {len(inner.blocks)} blocks, driven:",
      [p.spelling for p in inner.problems if p is not None])

reference, _, _ = mdf.nested_blocking(ref.ixc, ref.icc, ref.n_equality, ref.i_figure_merit,
                                      graph=machine_graph, switch_values=sw)
print("same as mdf.nested_blocking(...):", plan.graph == reference.graph)

the whole recipe:
   fixed_point_cut(.physics.proton_rate_density->^hat.physics.proton_rate_density, .physics.fusden_alpha_total->^ ...
   fixed_point_cut(.fwbs.f_ster_div_single->^hat.fwbs.f_ster_div_single @ ^problem.fwbs.f_ster_div_single)
   insert(.Constraint2, .Constraint16, .Constraint24, .Constraint8, .Constraint17, .Constraint18, .Constraint67,  ...
   nest_inside(.Opt)



answered at the top level: ['.Opt', '^problem.power.delta_eta_step']
interior of the optimiser's block: 112 blocks, driven: ['^problem.physics.profiles.ion_vol_avg_temperature', '^problem.stellarator.coils.intersect', '^problem.physics.proton_rate_density.cycle', '^problem.fwbs.f_ster_div_single']


same as mdf.nested_blocking(...): True


### Assign the drivers

`default_drivers` picks a driver by problem type: VMCON for the optimisation (the same
SQP PROCESS uses, via PyVMCON), fixed-point iteration for a cut, Newton for a root find.
`assign_drivers` attaches each to its problem in the graph. The nesting recorded above
is kept.

In [7]:
from cottax.evaluation.schedule import Schedule

from functional_process.cottax.mda import assign_drivers, default_drivers

runnable = assign_drivers(plan.graph, default_drivers(plan.graph, bounds=ref.bounds))
blocking = Blocking.scc(runnable)
for p in declared(runnable):
    print(f"{p.spelling:55s} {problem_kind(runnable[p]):12s} {driver_name(runnable[p])}")
print(f"\n{len(blocking.blocks)} top-level blocks; the optimiser's interior has {len(blocking.inner[blocking.index[OPT]].blocks)}")

^problem.stellarator.coils.intersect                    root-find    SeededNewtonDriver
^problem.physics.profiles.ion_vol_avg_temperature       fixed-point  PicardDriver
^problem.power.delta_eta_step                           fixed-point  PicardDriver
^problem.physics.proton_rate_density.cycle              fixed-point  PicardDriver
^problem.fwbs.f_ster_div_single                         fixed-point  PicardDriver
.Opt                                                    optimise     VmconDriver



47 top-level blocks; the optimiser's interior has 112


## The process

The DSM in run order. The optimiser's box encloses the whole analysis, and inside it
each iterated group has a box of its own. The DSM is written as an interactive page next to this notebook. In the page, hover a
cell for the variables it carries and click a box to fold it.

In [8]:
from functional_process.cottax.render_xdsm import SPELLING
from functional_process.cottax.visualization.grouping import render_grouped_dsm_html, structure_order

dsm = render_grouped_dsm_html(
    blocking, order=structure_order(blocking),
    title="stellarator_helias -- MDF: VMCON outside, the MDA nested inside it",
    file_name="dsm_mdf", outdir=str(HERE), write=True, formatter=SPELLING,
)
print("written:", dsm.path)

Using adapted ragraph from debug branch


written: /home/wrutten/projects/functional_PROCESS/PROCESS/functional_process/architecture_examples/mdf/dsm_mdf.html


## Run it

`mdf.assemble` is the runnable form of the structure above. The analysis is an inner
schedule; VMCON is called on its objective and constraints. Each evaluation is one
converged analysis, and the Jacobian is automatic differentiation through it. Three
steps, as `run_cold_matrix.solve_mdf` does them:

- `seed`: every input and every inner starting value from the input file;
- `prime`: run the analysis once and keep its answer as the starting point of the inner
  solves;
- `solve`: run VMCON on the iteration variables, then re-run the analysis at the answer.

In [9]:
from functional_process.cottax.run_cold_matrix import _recorder, _trace_tail
from functional_process.cottax.run_mdf_harness import MAX_ITER, TOLERANCE

problem = mdf.assemble(ref.ixc, ref.icc, ref.n_equality, ref.i_figure_merit,
                       graph=machine_graph, switch_values=sw)
shape = mdf.mdf_shape(problem)
print(f"outer problem: {shape['design']} design variables, {shape['conditions']} conditions "
      f"({shape['equalities']} equalities); inner MDA: {shape['inner_blocks']} blocks, "
      f"{shape['inner_driven']} driven, {shape['inner_unknowns']} unknowns\n")

began = time.perf_counter()
env = mdf.seed(problem, ref.cold)
env, primed = mdf.prime(problem, env)
print(f"seeded and primed in {time.perf_counter() - began:.1f} s (includes the MDA's compilation)")

trace = []
x, out, seconds = mdf.solve(problem, env, bounds=ref.bounds, callback=_recorder(trace),
                            tolerance=TOLERANCE, max_iter=MAX_ITER)
iterations, objf, max_eq, min_ie = _trace_tail(trace)
print(f"VMCON: {iterations} iterations in {seconds:.1f} s (first solve: includes compilation)")
print(f"objective {objf:.8f}   max|eq| {max_eq:.1e}   min ineq {min_ie:+.1e}")
print("design:", {i: round(float(v), 4) for i, v in zip(ref.ixc, x)})

outer problem: 8 design variables, 15 conditions (2 equalities); inner MDA: 158 blocks, 5 driven, 6 unknowns



seeded and primed in 2.7 s (includes the MDA's compilation)


VMCON: 40 iterations in 8.3 s (first solve: includes compilation)
objective 1.21844143   max|eq| 9.4e-11   min ineq -1.9e-09
design: {2: 4.7165, 3: 26.6444, 4: 5.7027, 6: 1.739178979491607e+20, 10: 1.048, 56: 31.8094, 59: 0.7177, 109: 0.0299}


The same solve is one line: `session.open_session(INPUT).mdf()`. That keeps the
assembly for repeated solves. `run_cold_matrix` runs it on every reference input file
beside SAND, and `reference_cold_matrix.txt` compares its numbers with PROCESS's own
VMCON run.

In [10]:
RESULT = {"iterations": iterations, "objf": objf, "max_eq": max_eq, "min_ie": min_ie,
          "design": {i: float(v) for i, v in zip(ref.ixc, x)},
          "converged": bool(max_eq < 1e-6 and iterations < MAX_ITER)}
RESULT

{'iterations': 40,
 'objf': 1.2184414321044577,
 'max_eq': 9.371348141939961e-11,
 'min_ie': -1.9102754933442156e-09,
 'design': {2: 4.716452313386795,
  3: 26.644439598217247,
  4: 5.70273553946853,
  6: 1.739178979491607e+20,
  10: 1.047952211971554,
  56: 31.80937214494472,
  59: 0.7176871442817623,
  109: 0.02992503865120092},
 'converged': True}